# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['NYJUHFZDDK', 'KHHKKKVRCJ'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[14, 25, 10, 21,  8,  6, 26,  4,  4, 11],
       [11,  8,  8, 11, 11, 11, 22, 18,  3, 10]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 11,  4,  4, 26,  6,  8, 21, 10, 25],
       [ 0, 10,  3, 18, 22, 11, 11, 11,  8,  8]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[11,  4,  4, 26,  6,  8, 21, 10, 25, 14],
       [10,  3, 18, 22, 11, 11, 11,  8,  8, 11]])>)


# 建立sequence to sequence 模型

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, input_shape=(None,))
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        完成sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好
        '''
        enc_emb = self.embed_layer(enc_ids)  # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)  # Encoder output and state

        dec_emb = self.embed_layer(dec_ids)  # shape(b_sz, len, emb_sz)
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state)  # Decoder output

        logits = self.dense(dec_out)  # shape(b_sz, len, v_sz)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        
        return [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.decoder_cell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
_, warmup_enc_x, warmup_dec_x, _ = get_batch(1, 20)
_ = model(warmup_enc_x, warmup_dec_x)
train(model, optimizer, seqlen=20)

d:\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


step 0 : loss 3.3119159
step 500 : loss 1.5367739
step 1000 : loss 0.95450336
step 1500 : loss 0.6634227
step 2000 : loss 0.5381219
step 2500 : loss 0.37293306


<tf.Tensor: shape=(), dtype=float32, numpy=0.2902819514274597>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('MIVZLYXLWH', 'HWLXYLZVIM'), ('DKNLRIBVLO', 'OLVBIRLNKD'), ('MDBORCIMDL', 'LDMICROBDM'), ('OFANJFTGTN', 'NTGTFJNAFO'), ('IMOTIXJISN', 'NSIJXITOMI'), ('ZBUBZBFKVY', 'YVKFBZBUBZ'), ('MNGAFIJBZK', 'KZBJIFAGNM'), ('MCCGBCIWVQ', 'QVWICBGCCM'), ('QUGKBFGSAU', 'UASGFBKGUQ'), ('PGYFQMMFIG', 'GIFMMQFYGP'), ('AHSUUOLTMH', 'HMTLOUUSHA'), ('YBMDNINPBF', 'FBPNINDMBY'), ('GNCJREZJMI', 'IMJZERJCNG'), ('THJSYDGMUO', 'OUMGDYSJHT'), ('GKIWSJXVIU', 'UIVXJSWIKG'), ('IFNGBIOTIO', 'OITOIBGNFI'), ('KTIZWPQZEW', 'WEZQPWZITK'), ('APHOHWZGDA', 'ADGZWHOHPA'), ('EZYHYGAHWG', 'GWHAGYHYZE'), ('LVOCPYZJWJ', 'JWJZYPCOVL'), ('XWACLACQPO', 'OPQCALCAWX'), ('YGZQLYLEGZ', 'ZGELYLQZGY'), ('OOCGATWEED', 'DEEWTAGCOO'), ('NBYMOCSLZV', 'VZLSCOMYBN'), ('GFFBTWOIPN', 'NPIOWTBFFG'), ('VRGEKVVBID', 'DIBVVKEGRV'), ('EHOYYRRLTW', 'WTLRRYYOH